# Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics.pairwise import rbf_kernel
import itertools
import time
from tqdm.auto import tqdm

# For NN: Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using acceleration device: {device}")

# General settings and data import

In [ ]:
SEED=118

#from google.colab import drive # Only if running from Colab
#drive.mount('/content/drive') # Only if running from Colab
DATA_DIR = "/kaggle/working" # Change to appropriate local path
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True) # If the folder doesn't exist, create it

x_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_train_houses.csv")
y_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/y_train_houses.csv")
x_test_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_test_houses.csv")

features=x_train_df.columns[1:].tolist()
y_tr_raw=y_train_df.iloc[:, 1].values

# IDA and EDA

In [ ]:
x_train = x_train_df[features]
x_test = x_test_df[features]

# 1. Look for missing data
print(x_train.isna().sum())
print(x_test.isna().sum()) # there is missing data in total_bedrooms

# 2. Numerical variables' distributions
print(x_train.describe())
x_train.assign(Price=y_tr_raw).hist(bins=40, figsize=(12,8))
plt.tight_layout()
plt.show()

# 3. Plot house on a map based on latitude/longitude
cities=np.array([[34.1141,-118.4068],   # Los Angeles
                 [38.5677,-121.4685],   # Sacramento
                 [32.8313,-117.1222],   # San Diego
                 [37.7558,-122.4449],   # San Francisco
                 [37.3012,-121.8480]])  # San Jose
# source: https://simplemaps.com/data/us-cities
city_names = ["Los Angeles", "Sacramento", "San Diego", "San Francisco", "San Jose"]

plt.figure(figsize=(8,8))
plt.scatter(x_train["longitude"], x_train["latitude"], c=y_tr_raw, alpha=0.4, s=7)
plt.colorbar(label="Price")
plt.scatter(cities[:, 1], cities[:, 0], color="red", edgecolor="white", s=150, marker="*")
for i, name in enumerate(city_names):
    txt = plt.text(cities[i, 1] + 0.15, cities[i, 0] - 0.05, name, color="black", fontsize=10, fontweight="bold")
    txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground="white")])
plt.gca().set_aspect("equal")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()
# Expensive houses near the big cities. An engineered feature that measures distance from them could have improving effects?

# 4. Mean price by Ocean proximity
sns.boxplot(data=x_train.assign(Price=y_tr_raw), x="ocean_proximity", y="Price",
            order=x_train.assign(Price=y_tr_raw).groupby("ocean_proximity")["Price"].mean().sort_values(ascending=False).index)
plt.show()
# Expensive houses are located near the coast

# 5. Correlations
heatmap = sns.heatmap(x_train.assign(Price=y_tr_raw).select_dtypes("number").corr(), cmap="coolwarm", annot=True)
## Correlations with response variable "Price"
corr_price = x_train.assign(Price=y_tr_raw).select_dtypes("number").corr()["Price"].sort_values(key=abs, ascending=False)
corr_price
# most correlated variable: median_income

* There is missing data in total_bedrooms
* Price and house_median_age have some hard caps, respectively at 500k $ and 52 years
* total_rooms, total_bedrooms, population and households are highly skewed
* median_income is skewed but not as much
* Houses are more expensive near the 5 main cities of California (Los Angeles, Sacramento, San Diego, San Francisco, San Jose)
* In general, houses near the coast are more expensive, but exceptions exist
* median_income is the most correlated variable





# Feature engineering

In [ ]:
# Histograms: the totals measure district size -> per-household ratios describe the typical home
def ratios(d):
    return d.assign(rooms_per_household=d["total_rooms"]/d["households"],
                    population_per_household=d["population"]/d["households"],
                    bedrooms_per_room=d["total_bedrooms"]/d["total_rooms"],
                    #income_per_room=d["median_income"]/d["total_rooms"]
                   )

def new_features(d):
    return d.assign(income_x_age=d["median_income"]*d["housing_median_age"], # Old houses, high income -> possible wealthy area
                    income_per_room=d["median_income"]/d["rooms_per_household"], # income per unit of housing space
                   )

# Lat/long as a smooth 3D position on the sphere, to account for the earth's curvature
def sphere_coords(d):
    lat=np.radians(d["latitude"])
    lon=np.radians(d["longitude"])
    return d.assign(x_coord=np.cos(lat)*np.cos(lon),
                    y_coord=np.cos(lat)*np.sin(lon),
                    z_coord=np.sin(lat))

# Map: expensive houses are near the big cities -> distance to the 2 closest main cities
def dist_city(d):
    dists = cdist(d[["latitude", "longitude"]].to_numpy(), cities)
    sorted_dists = np.sort(dists, axis=1)
    return d.assign(dist_nearest_city=sorted_dists[:, 0],
                    dist_2nd_nearest_city=sorted_dists[:, 1])

# Map: split CA into k-means clusters, and score each district by its similarity (closeness) to each cluster centroid
# Gamma and n_clusters chosen through grid search (not reported here)
def kmeans_pipeline(gamma=15, n_clusters=75):
    kmeans = KMeans(n_clusters, n_init=10, random_state=SEED)
    return make_pipeline(kmeans, FunctionTransformer(lambda d: np.exp(-gamma * d ** 2)))

* total_rooms, total_bedrooms and population are measures for the entire district -> per-household ratios describe the typical home
* bedroom/room and room/person measure the "type" of house and how much space there is for each person
* dist_nearest_city measures the distance from two of the 5 major cities in California (Los Angeles, Sacramento, San Diego, San Francisco, San Jose)
* Latitude and Longitude are transformed to a XYZ coordinate system, to account for Earth's curvature

# Linear regression, without and with polynomial features

In [ ]:
cv=KFold(10, shuffle=True, random_state=SEED) # same folds for every model -> paired comparisons
scoring={"mse":"neg_mean_squared_error", "mae":"neg_mean_absolute_error", "r2":"r2"}

# steps: row-wise feature functions, applied first (they only use their own row, so no leakage).
# Everything that is fitted (imputer, scalers, ridge) lives inside the pipeline,
# so cross_validate refits it on each training fold only.
def build(steps, model, degree=1, kmeans=False, gamma=15, n_clusters=75):
    num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)    
    preprocessing = ColumnTransformer(
        [("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))] +
        ([("km", kmeans_pipeline(gamma, n_clusters), ["latitude", "longitude"])] if kmeans else []),
        remainder=num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

def cross_val_all(models, n_jobs=-1):
    return {name:cross_validate(m, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=n_jobs)
            for name, m in models.items()}

# train_* = in-sample, cv_* = out-of-sample (mean over the 10 folds).
# rmse_gain: positive = better than the reference row. folds_better: in how many of the 10 folds it beat the reference.
# Reference = the row above, or the first row if vs_first=True (the first row compares with itself).
def summarise(res, vs_first=False):
    names=list(res)
    mse={n:-res[n]["test_mse"] for n in names}
    out=pd.DataFrame(index=names)
    out["train_rmse"]=[np.sqrt(-res[n]["train_mse"].mean()) for n in names]
    out["train_r2"]=[res[n]["train_r2"].mean() for n in names]
    out["cv_rmse"]=[np.sqrt(mse[n].mean()) for n in names]
    out["cv_mae"]=[-res[n]["test_mae"].mean() for n in names]
    out["cv_r2"]=[res[n]["test_r2"].mean() for n in names]
    ref=[names[0] if vs_first else names[max(i-1, 0)] for i in range(len(names))]
    out["rmse_gain"]=[out.loc[r, "cv_rmse"] - out.loc[n, "cv_rmse"] for n, r in zip(names,ref)]
    out["folds_better"]=[int((mse[n] < mse[r]).sum()) for n, r in zip(names, ref)]
    return out.round(3)

In [ ]:
# Row 1 is point a (plain linear regression); each following row adds one block to the row above.
sets={"1 baseline":([], False),
      "2 +ratios":([ratios], False),
      "3 +new_features":([ratios, new_features], False),
      "4 +sphere_coords":([ratios, new_features, sphere_coords], False),
      "5 +dist_city":([ratios, new_features, sphere_coords, dist_city], False),
      "6 +kmeans clusters":([ratios, new_features, sphere_coords, dist_city], True)}

ladder=cross_val_all({name:build(steps, LinearRegression(), kmeans=km) for name, (steps, km) in sets.items()})
summarise(ladder)

Each feature reduces the RMSE, i.e. improves the accuracy of the model. This means that these feature are adequate for this dataset.

In [ ]:
# EDIT after reading the ladder: keep only the blocks that helped
keep_steps=[ratios, new_features, dist_city, sphere_coords]
keep_kmeans = True

# Features to test
feats = {"raw": ([], False), "engineered": (keep_steps, False), "engineered_kmeans": (keep_steps, True)}

In [ ]:
# Polynomial terms on all original numeric columns; dummies and k-means distances enter linearly.
# Ridge is included otherwise the error explodes at higher degrees.
# Degree 1 is included as a ridge control, so any gain at degree 2-3 is due to the polynomial terms.
def build_poly(steps, model, degree=1, kmeans=True, gamma=15, n_clusters=75):
    original_numeric = ["housing_median_age", "total_rooms", "total_bedrooms", "population",
                    "households", "median_income", "latitude", "longitude"]
    
    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    original_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                          PolynomialFeatures(degree, include_bias=False), StandardScaler())
    other_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    parts = [("original_num_poly", original_num_pipeline, original_numeric),
             ("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))]
    if kmeans:
        parts.append(("km", kmeans_pipeline(gamma, n_clusters), ["latitude", "longitude"]))
    preprocessing = ColumnTransformer(parts, remainder=other_num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

ridge=RidgeCV(alphas=np.logspace(-2, 4, 25))
poly_models = {}
for label, (steps, km) in feats.items():
    for degree in [1, 2, 3]:
        poly_models[f"ridge deg{degree} | {label}"] = build_poly(steps, ridge, degree=degree, kmeans=km)

poly = cross_val_all(poly_models)
summarise(poly, vs_first=True) # every row compared with "ridge deg1 | baseline"

Best model is a linear model on the engineered_kmeans dataset, but 2nd and 3rd degree polynomials are not much worse.
On other sets, adding polynomial features actually improves RMSE

In [ ]:
# Price is capped at ~500k in the data (Price histogram) -> keep predictions inside the training range
def submit(model, filename):
    model.fit(x_train, y_tr_raw) # median, scalers, k-means and alpha are fitted on all of x_train only
    pred=model.predict(x_test)
    print(filename, "non-positive predictions before clipping:", (pred<=0).sum())
    pred=np.clip(pred, y_tr_raw.min(), y_tr_raw.max())
    pd.DataFrame({"ID":x_test_df.iloc[:, 0], "Price":pred}).to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

submit(build([], LinearRegression()), "submission_1a.csv") # point a
submit(build_poly(keep_steps, ridge, 1, keep_kmeans), "submission_1b.csv") # point b, EDIT: best row of the table

# Neural network

In [ ]:
# Utility functions
def to_tensor(array):
    return torch.tensor(np.asarray(array), dtype=torch.float32, device=device)

def parse_architecture(arch_str):
    # converts a string like "128-64" into a tuple (128, 64)
    if arch_str == "linear":
        return ()
    return tuple(int(w) for w in arch_str.split("-"))

# Preprocessing (imputer, scalers, k-means, one-hot) is fitted on the fitting rows ONLY,
# then applied to every other set (early-stopping, validation and test rows)
def prep_fit(steps, X_fit, y_fit, *other_splits, kmeans=False, gamma=15, n_clusters=75):
    preprocessor = build(steps, "passthrough", kmeans=kmeans, gamma=gamma, n_clusters=n_clusters)
    X_fit_transformed = preprocessor.fit_transform(X_fit, y_fit)
    other_transformed = [preprocessor.transform(X) for X in other_splits]
    return [X_fit_transformed] + other_transformed

# 1. Construct the network: made it flexible to allow testing for multiple architectures
def build_network(n_inputs, hidden_widths, activation, dropout, use_batchnorm=False):
    # hidden_widths=() gives linear regression
    layers=[]
    in_features=n_inputs

    for width in hidden_widths:
        layers.append(nn.Linear(in_features, width))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(ACTIVATIONS[activation]())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_features=width

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)

# 2. Train the network
def train_network(X_train, y_train, X_val, y_val, hidden_widths=(128, 64), activation="relu",
                  learning_rate=1e-3, dropout=0.1, weight_decay=1e-4, batch_size=256, use_batchnorm=False,
                  seed=SEED, max_epochs=300, patience=30):
    # Train with MSE loss and AdamW. LR halves on plateau; stops early on validation RMSE and restores the best epoch's weights
    torch.manual_seed(seed)

    # standardize the target using training statistics only
    y_mean, y_std = y_train.mean(), y_train.std()
    X_train_t = to_tensor(X_train)
    X_val_t = to_tensor(X_val)
    y_train_t = to_tensor((y_train-y_mean)/y_std).unsqueeze(1)

    net = build_network(X_train_t.shape[1], hidden_widths, activation, dropout, use_batchnorm).to(device)
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
    loss_fn = nn.MSELoss()

    def compute_rmse(X, y_true):
        # RMSE in dollars, with dropout/batchnorm turned off
        net.eval()
        with torch.no_grad():
            preds = net(X).squeeze(1).cpu().numpy() * y_std + y_mean
        return np.sqrt(np.mean((preds - y_true) ** 2))

    best_val_rmse = np.inf
    best_epoch = 0
    best_weights = None
    history = []
    lrs = [] # learning rate used in each epoch (needed to repeat this training on 100% of the rows)

    for epoch in range(max_epochs):
        lrs.append(optimizer.param_groups[0]["lr"])
        net.train()
        shuffled_idx = torch.randperm(len(X_train_t), device=device)

        for start in range(0, len(shuffled_idx), batch_size):
            batch_idx = shuffled_idx[start:start + batch_size]
            if len(batch_idx) < 2:
                continue  # batchnorm needs at least 2 rows
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_t[batch_idx]), y_train_t[batch_idx])
            loss.backward()
            optimizer.step()

        train_rmse = compute_rmse(X_train_t, y_train)
        val_rmse = compute_rmse(X_val_t, y_val)
        history.append((train_rmse, val_rmse))
        scheduler.step(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_weights = {k: v.clone() for k, v in net.state_dict().items()}
        elif epoch - best_epoch >= patience:
            break

    net.load_state_dict(best_weights)
    return {"net": net, "y_mean": y_mean, "y_std": y_std, "best_epoch": best_epoch, "history": history, "lrs": lrs}

# 2b. Same training with no validation set (final model on 100% of the rows): runs exactly n_epochs and uses the
# learning rate lrs[epoch] at each epoch, both copied from an early-stopped train_network run with the same configuration
def train_network_full(X_train, y_train, n_epochs, lrs, hidden_widths, activation, learning_rate, dropout,
                       weight_decay, batch_size, use_batchnorm, seed=SEED):
    torch.manual_seed(seed)
    y_mean, y_std = y_train.mean(), y_train.std()
    X_train_t = to_tensor(X_train)
    y_train_t = to_tensor((y_train-y_mean)/y_std).unsqueeze(1)

    net = build_network(X_train_t.shape[1], hidden_widths, activation, dropout, use_batchnorm).to(device)
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    for epoch in range(n_epochs):
        for group in optimizer.param_groups:
            group["lr"] = lrs[epoch]
        net.train()
        shuffled_idx = torch.randperm(len(X_train_t), device=device)
        for start in range(0, len(shuffled_idx), batch_size):
            batch_idx = shuffled_idx[start:start + batch_size]
            if len(batch_idx) < 2:
                continue  # batchnorm needs at least 2 rows
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_t[batch_idx]), y_train_t[batch_idx])
            loss.backward()
            optimizer.step()

    return {"net": net, "y_mean": y_mean, "y_std": y_std}

# 3. Use the network to make predictions
def predict(result, X):
    result["net"].eval()
    with torch.no_grad():
        preds = result["net"](to_tensor(X)).squeeze(1).cpu().numpy()
    return preds * result["y_std"] + result["y_mean"]

# 4. Tuning
def run_grid(configs, feature_set_name):
    X_train, X_val = data[feature_set_name]
    results_rows = []
    histories = []

    for config in tqdm(configs):
        start_time = time.time()
        result = train_network(X_train, ya, X_val, yb, **config)
        pred_train = predict(result, X_train)
        pred_val = predict(result, X_val)
        arch_label = "-".join(map(str, config["hidden_widths"])) or "linear"
        other_params = {k: v for k, v in config.items() if k != "hidden_widths"}
        results_rows.append({
            "feat": feature_set_name,
            "arch": arch_label,
            **other_params,
            "best_epoch": result["best_epoch"] + 1,
            "train_rmse": np.sqrt(mean_squared_error(ya, pred_train)),
            "val_rmse": np.sqrt(mean_squared_error(yb, pred_val)),
            "val_mae": mean_absolute_error(yb, pred_val),
            "val_r2": r2_score(yb, pred_val),
            "secs": time.time() - start_time,
        })
        histories.append(result["history"])
    return pd.DataFrame(results_rows), histories

# For visualization purposes:
def show_grid(df, rows, cols, value="val_rmse"):
    # Pivot into a rows x cols table of validation RMSE, colored (darker=lower error)
    return df.pivot_table(index=rows, columns=cols, values=value).round(0).style.background_gradient(cmap="viridis_r", axis=None)

def plot_curves(histories, titles):
    # Diagnostic plot for convergence
    n_plots = len(histories)
    n_cols = 4
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    lowest_val_rmse = min(min(val for _, val in hist) for hist in histories)

    for ax, hist, title in zip(axes, histories, titles):
        hist = np.array(hist)
        ax.plot(hist[:, 0], label="train")
        ax.plot(hist[:, 1], label="validation")
        ax.set_title(title)
        ax.set_ylim(lowest_val_rmse * 0.6, lowest_val_rmse * 2)

    # hide unused subplot slots
    for ax in axes[n_plots:]:
        ax.axis("off")

    axes[0].legend()
    fig.supxlabel("epoch")
    fig.supylabel("RMSE ($)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Tuning functions
def to_cfg(row):
    # one row / dict of SPACE values -> keyword arguments for train_network
    return dict(hidden_widths=parse_architecture(row["arch"]), activation=row["activation"],
                learning_rate=float(row["learning_rate"]), dropout=float(row["dropout"]),
                weight_decay=float(row["weight_decay"]), batch_size=int(row["batch_size"]),
                use_batchnorm=bool(row["use_batchnorm"]))

def sample_configs(space, n, seed):
    # same random configurations every time (seeded), so a resumed run continues the same list
    rng = np.random.default_rng(seed)
    seen = set()
    configs = []
    while len(configs) < n:
        c = {k: v[rng.integers(len(v))] for k, v in space.items()}
        key = tuple(c.values())
        if key not in seen:
            seen.add(key)
            configs.append(c)
    return configs

def cached_grid(filename, configs, feat, every=10):
    # run_grid with a CSV checkpoint every `every` networks; resumes after a Colab disconnect
    path = os.path.join(OUTPUT_DIR, filename)
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    for i in range(len(done), len(configs), every):
        part, _ = run_grid(configs[i:i + every], feat)
        done = pd.concat([done, part], ignore_index=True)
        done.to_csv(path, index=False)
    return done

def screen_survivors(screen, space, keep):
    # best `keep[dim]` values of each hyperparameter, ranked by mean validation RMSE
    survivors = {}
    for dim in space:
        stats = screen.groupby(dim)["val_rmse"].agg(mean="mean", median="median", best="min", n="count")
        stats = stats.sort_values("median")
        display(dim, stats.round(0))
        survivors[dim] = list(stats.index[:keep[dim]])
    return survivors

def recheck_top(final, feat, n_recheck, space, seeds):
    # re-run the best `n_recheck` networks with new seeds, since a single run can be noisy
    rows = []
    for _, row in final.sort_values("val_rmse").head(n_recheck).iterrows():
        repeats, _ = run_grid([{**to_cfg(row), "seed": s} for s in seeds], feat)
        rows.append(dict(
            **{k: row[k] for k in space},
            step2_mean=row["val_rmse"],
            mean_seeds=repeats["val_rmse"].mean(),
            sd_seeds=repeats["val_rmse"].std(),
            mean_best_epoch=repeats["best_epoch"].mean(),
        ))
    return pd.DataFrame(rows).sort_values("mean_seeds")

def tune_nn(feat, space, n_screen, keep, max_final, n_recheck, recheck_seeds, seed, tag, step2_seeds):
    # step 1: broad random screen over the whole space
    screen = cached_grid(f"nn_screen_{tag}.csv", [to_cfg(c) for c in sample_configs(space, n_screen, seed)], feat)
    survivors = screen_survivors(screen, space, keep)
    print("surviving values:", survivors)

    # step 2: full grid over the survivors only (subsampled if larger than max_final)
    combos = list(itertools.product(*survivors.values()))
    if len(combos) > max_final:
        pick = np.random.default_rng(seed).choice(len(combos), max_final, replace=False)
        combos = [combos[i] for i in np.sort(pick)]
        
    configs=[to_cfg(dict(zip(survivors, c))) for c in combos]
    runs=[cached_grid(f"nn_final_{tag}_seed{s}.csv", [{**c, "seed":s} for c in configs], feat) for s in step2_seeds]
    # one row per configuration, every result column averaged over the step-2 seeds
    metric_cols=["best_epoch", "train_rmse", "val_rmse", "val_mae", "val_r2", "secs"]
    final=pd.concat(runs).groupby(list(space), as_index=False, sort=False)[metric_cols].mean()
    
    display(final.sort_values("val_rmse").head(15).round(3))
    display(show_grid(final, "arch", ["activation", "learning_rate"]))
    display(show_grid(final, "dropout", ["weight_decay", "batch_size"]))

    # step 3: re-run the best few with new seeds, since a single run is noisy
    recheck = recheck_top(final, feat, n_recheck, space, recheck_seeds)
    display(recheck.round(3))
    return to_cfg(recheck.iloc[0]), screen, final, recheck

In [ ]:
def cross_validate_nn(cfg, feat, seeds=(89, 233, 1597)):
    # Honest CV: each fold trains a fresh ensemble of `seeds` networks, with preprocessing fit on that
    # fold's training rows only. Early stopping watches a held-out 10% split (`stop`), never the
    # validation fold (`val`), so nothing about the validation fold leaks into training decisions.
    # Keys match cross_validate's naming (train_mse/test_mse/...) so summarise() works on both.
    steps, km = feats[feat]
    y = y_tr_raw
    fold_scores = dict(train_mse=[], test_mse=[], test_mae=[], train_r2=[], test_r2=[])
    test_preds = []
    oof_preds = np.zeros(len(y))

    for train_idx, val_idx in tqdm(list(cv.split(x_train))):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val, X_test = prep_fit(
            steps, x_train.iloc[fit_idx], y[fit_idx], x_train.iloc[stop_idx],
            x_train.iloc[val_idx], x_test, kmeans=km)

        fit_preds, val_preds, test_fold_preds = [], [], []
        for seed in seeds:
            result = train_network(X_fit, y[fit_idx], X_stop, y[stop_idx], seed=seed, **cfg)
            fit_preds.append(predict(result, X_fit))
            val_preds.append(predict(result, X_val))
            test_fold_preds.append(predict(result, X_test))

        fit_pred = np.mean(fit_preds, axis=0)  # ensemble = average of the seeds
        val_pred = np.mean(val_preds, axis=0)
        test_preds.append(np.mean(test_fold_preds, axis=0))
        oof_preds[val_idx] = val_pred

        fold_scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        fold_scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        fold_scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        fold_scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        fold_scores["test_r2"].append(r2_score(y[val_idx], val_pred))

    fold_scores = {k: np.array(v) for k, v in fold_scores.items()}
    return fold_scores, np.mean(test_preds, axis=0), oof_preds

In [ ]:
# Features to test
ACTIVATIONS = {"relu":nn.ReLU, "leaky_relu":nn.LeakyReLU, "elu":nn.ELU, "silu":nn.SiLU, "gelu":nn.GELU, "tanh":nn.Tanh}
FINAL_FEATURE_SET = "engineered_kmeans"

# Tuning configuration
"""
Step 1: random search over everything at once; for each value of each hyperparameter we look at the average
  validation RMSE of all the networks that used it. Therefore, a value that is consistently worse is dropped.
Step 2: full search over the "surviving" values (if < MAX_FINAL, otherwise another random search).
Step 3: the best networks are re-run with new seeds and the best average is chosen as the final configuration.
Both steps are saved to CSV in OUTPUT_DIR and resume after a Colab disconnect; delete the CSV files to repeat a step.
"""

FIXED_CFG = dict(hidden_widths=(256,128), activation="relu", learning_rate=5e-3, dropout=0.2,
                 weight_decay=1e-3, batch_size=128, use_batchnorm=True) # Best config found by the tuning below
RUN_TUNING = True # False: skip the search and use FIXED_CFG
N_SCREEN = 180 # Networks to test in step 1
KEEP = dict(arch=3, activation=3, learning_rate=2, dropout=2,
            weight_decay=1, batch_size=1, use_batchnorm=1) # How many values of each hyperparameter survive step 1
MAX_FINAL = 120 # Maximum number of networks to test in step 2
N_RECHECK = 5 # Top networks from step 2, to test with new seeds at the end to eliminate noise
RECHECK_SEEDS = (9, 28, 630, 2026, 40121, 271828, 7778777)
STEP2_SEEDS=(31337, 65537, 104729)

# Hyperparameter candidates
SPACE = dict(
    arch=["256", "64-64", "128-64", "256-128", "128-64-32", "256-128-64"],
    activation=["relu", "leaky_relu", "elu", "silu", "gelu", "tanh"],
    learning_rate=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    dropout=[0, 0.05, 0.1, 0.2, 0.3, 0.4],
    weight_decay=[0, 1e-3, 1e-2, 1e-1, 1],
    batch_size=[64, 128, 256, 512],
    use_batchnorm=[False, True],
)

In [ ]:
# Divide into train/validation split for tuning (80/20, random)
idx_tr, idx_va = train_test_split(np.arange(len(y_tr_raw)), test_size=0.2, random_state=SEED)
xa, ya = x_train.iloc[idx_tr], y_tr_raw[idx_tr]
xb, yb = x_train.iloc[idx_va], y_tr_raw[idx_va]

data = {name: tuple(prep_fit(steps, xa, ya, xb, kmeans=km)) for name, (steps, km) in feats.items()}
print({name: d[0].shape[1] for name, d in data.items()}) # number of input columns per feature set

# reference: the linear model (best features) on this exact split
Xa, Xb = data[FINAL_FEATURE_SET]
lm = LinearRegression().fit(Xa, ya)
print("linear model on this split | train RMSE:", round(np.sqrt(mean_squared_error(ya, lm.predict(Xa)))),
      "| validation RMSE:", round(np.sqrt(mean_squared_error(yb, lm.predict(Xb)))))

base = dict(hidden_widths=(128,64), activation="relu", learning_rate=1e-3, dropout=0.1, weight_decay=1e-4, batch_size=256) # defaults, overridden in each stage

In [ ]:
# No dropout / weight decay here, so overfitting is visible. Early stopping is always on.
ladder_widths = [(), (256,), (128,64), (256,128), (256,128,64), (512,256,128), (256,128,64,32)]
nn_ladder, nn_hists = run_grid([{**base, "hidden_widths":w, "dropout":0, "weight_decay":0} for w in ladder_widths], FINAL_FEATURE_SET)
display(nn_ladder[["arch", "best_epoch", "train_rmse", "val_rmse", "val_mae", "val_r2", "secs"]].round(3))
plot_curves(nn_hists, nn_ladder["arch"])

The "linear" network has a similar result to the linear regression fit, confirming that a NN without nodes is equivalent to a linear model.
We can see that the biggest improvement comes when adding the first layer to the network, then all gains are marginals, plateauing around 45000 val_RMSE (.842/.845 val_R2).
From these results, we can avoid testing 4-layer networks, as they are computationally expensive to train and they do not improve enough the prediction task.

In [ ]:
fs = pd.concat([run_grid([{**base, "seed":s} for s in range(3)], name)[0] for name in feats])
display(fs.groupby("feat")[["train_rmse", "val_rmse", "val_mae", "val_r2"]].mean().round(3))

This test confirms that the engineered features are useful for the NN as well, and that k-means distances help a bit more. 
The final model will use the engineered features + k-means distances.

In [ ]:
# Tuning
if RUN_TUNING:
    best_cfg, screen, final, recheck = tune_nn(
        feat=FINAL_FEATURE_SET, space=SPACE, n_screen=N_SCREEN, keep=KEEP, max_final=MAX_FINAL,
        n_recheck=N_RECHECK, recheck_seeds=RECHECK_SEEDS, seed=SEED, tag=FINAL_FEATURE_SET, step2_seeds=STEP2_SEEDS)
else:
    best_cfg = FIXED_CFG

print(best_cfg)

In [ ]:
nn_cv, nn_test, nn_oof = cross_validate_nn(best_cfg, FINAL_FEATURE_SET)

# 3-way comparison on the same 10 folds, every row compared with the first one:
# 1. linear regression on the final feature set (engineered + k-means)
# 2. the best ridge/polynomial model of the table above (already cross-validated on these folds, so it is not refitted)
# 3. the NN (3-seed ensemble)
# cross_validate_nn returns the same format as cross_validate, so summarise() works on all three
lin = cross_val_all({"linear regression | engineered_kmeans": build(keep_steps, LinearRegression(), kmeans=keep_kmeans)})
best_poly = summarise(poly)["cv_rmse"].idxmin()
display(summarise({**lin, best_poly: poly[best_poly], "NN (3-seed ensemble)": nn_cv}, vs_first=True))

In [ ]:
# Final NN (90/10): preprocessing fitted on ALL training rows, so test districts see every labelled neighbour
# (the CV folds above only see 81% of the training rows as neighbours). 90/10 split only for early stopping, 5 seeds averaged.
# This is exactly the training recipe validated in the CV above.
steps, km = feats[FINAL_FEATURE_SET]
X_all, X_test_all = prep_fit(steps, x_train, y_tr_raw, x_test, kmeans=km)
fit_idx, stop_idx = train_test_split(np.arange(len(y_tr_raw)), test_size=0.1, random_state=SEED)

def save_submission(test_pred, filename):
    print(filename, "| non-positive predictions before clipping:", (test_pred <= 0).sum(),
          "| above the training max:", (test_pred > y_tr_raw.max()).sum())
    pred = np.clip(test_pred, y_tr_raw.min(), y_tr_raw.max())
    pd.DataFrame({"ID": x_test_df.iloc[:, 0], "Price": pred}).to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

test_preds, early_runs = [], []
for seed in tqdm(range(5)):
    result = train_network(X_all[fit_idx], y_tr_raw[fit_idx], X_all[stop_idx], y_tr_raw[stop_idx], seed=seed, **best_cfg)
    test_preds.append(predict(result, X_test_all))
    n_ep = result["best_epoch"] + 1
    early_runs.append((seed, n_ep, result["lrs"][:n_ep])) # reused by the 100% model below
    print(f"seed {seed}: best epoch {n_ep}, learning rate halved {int(np.sum(np.diff(result['lrs'][:n_ep]) < 0))} times")

save_submission(np.mean(test_preds, axis=0), "submission_1c_90.csv")
save_submission(nn_test, "submission_1c_cv.csv") # alternative: average of the 30 networks of the CV above (10 folds x 3 seeds)

In [ ]:
# Final NN (100%): the same recipe repeated on ALL training rows. No rows are left for early stopping, so each seed
# copies from its 90/10 run above the number of epochs and the learning rate of every epoch
# -> same training as the validated one, just with 11% more data. 5 seeds averaged.
test_preds = []
for seed, n_ep, lrs in tqdm(early_runs):
    result = train_network_full(X_all, y_tr_raw, n_ep, lrs, seed=seed, **best_cfg)
    test_preds.append(predict(result, X_test_all))

save_submission(np.mean(test_preds, axis=0), "submission_1c_full.csv")

# APPENDIX - tests only, NOT part of the analysis: k-means `n_clusters` and `gamma`

Everything below only tests the two k-means settings (`n_clusters`, `gamma`); nothing above depends on it.
It needs the cells above to have been run (data, features, `build`, `build_poly`, `prep_fit`, `train_network`, `cv`, `best_cfg`).

* `gamma` sets how fast the similarity `exp(-gamma * d^2)` to a centroid decays with the distance `d` (in degrees).
  The tables show it also as a *half-width*: the distance (km) at which the similarity drops to 0.5.
* Every pair `(n_clusters, gamma)` is cross-validated on the same 10 folds as the rest of the notebook, with the same pipeline as the final model.
  `rmse_gain` / `folds_better` compare each pair with the current setting (`n_clusters=75, gamma=15`).
* Results are saved to `OUTPUT_DIR/kmeans_tests/` after every pair. Re-running a cell skips the pairs already done
  (so a Colab disconnect loses at most one pair); delete the CSV to start again.

In [ ]:
# ---------- TEST 1: n_clusters x gamma, linear model (ridge) ----------
# Same pipeline as submission 1b (build_poly, degree 1, engineered + k-means); only n_clusters and gamma change.
# RidgeCV instead of LinearRegression: with hundreds/thousands of (very similar) similarity columns OLS breaks down.
KM_TEST_DIR = os.path.join(OUTPUT_DIR, "kmeans_tests")
os.makedirs(KM_TEST_DIR, exist_ok=True)
METRICS = ["train_mse", "test_mse", "test_mae", "train_r2", "test_r2"]
REF = (75, 15)  # current setting (n_clusters, gamma): every other pair is compared with it

def half_width_km(gamma):
    # distance at which exp(-gamma * d^2) = 0.5, with 1 degree ~ 111 km
    return 111 * np.sqrt(np.log(2) / gamma)

def run_km_grid(filename, grid, score_fn):
    # score_fn(n_clusters, gamma) -> per-fold scores in cross_validate format. One CSV row per fold, saved after every pair
    path = os.path.join(KM_TEST_DIR, filename)
    done = pd.read_csv(path) if os.path.exists(path) else None
    for k, g in grid:
        if done is not None and ((done["n_clusters"] == k) & (done["gamma"] == g)).any():
            continue
        start_time = time.time()
        scores = score_fn(k, g)
        rows = pd.DataFrame({"n_clusters": k, "gamma": g, "fold": range(len(scores["test_mse"])),
                             **{m: scores[m] for m in METRICS}})
        done = rows if done is None else pd.concat([done, rows], ignore_index=True)
        done.to_csv(path, index=False)
        print(f"n_clusters={k}, gamma={g}: cv_rmse={np.sqrt(-np.mean(scores['test_mse'])):,.0f} ({time.time() - start_time:.0f}s)")
    return done

def km_summary(df, ref=REF):
    # summarise() table, one row per (n_clusters, gamma), sorted by cv_rmse; rmse_gain and folds_better are vs `ref`
    pairs = sorted(dict.fromkeys(zip(df["n_clusters"], df["gamma"])), key=lambda p: p != tuple(ref))  # ref first
    res = {}
    for k, g in pairs:
        d = df[(df["n_clusters"] == k) & (df["gamma"] == g)].sort_values("fold")
        res[f"k={k}, gamma={g:g}"] = {m: d[m].to_numpy() for m in METRICS}
    out = summarise(res, vs_first=True)
    out.insert(0, "n_clusters", [k for k, _ in pairs])
    out.insert(1, "gamma", [g for _, g in pairs])
    out.insert(2, "half_width_km", [round(half_width_km(g), 1) for _, g in pairs])
    return out.sort_values("cv_rmse")

def km_heatmap(summary, value="cv_rmse"):
    # n_clusters x gamma table (darker = better)
    table = summary.pivot_table(index="n_clusters", columns="gamma", values=value)
    table.columns = [f"gamma={g:g} (~{half_width_km(g):.0f} km)" for g in table.columns]
    cmap = "viridis" if value.endswith("r2") else "viridis_r"
    return table.round(3 if value.endswith("r2") else 0).style.background_gradient(cmap=cmap, axis=None)

def km_plot(summary, title, filename):
    # CV RMSE against n_clusters, one line per gamma (light = wide similarity, dark = narrow); saved as PNG for the report
    fig, ax = plt.subplots(figsize=(8, 5))
    gammas = sorted(summary["gamma"].unique())
    for g, color in zip(gammas, plt.cm.Blues(np.linspace(0.35, 1, len(gammas)))):
        s = summary[summary["gamma"] == g].sort_values("n_clusters")
        ax.plot(s["n_clusters"], s["cv_rmse"], marker="o", markersize=6, linewidth=2, color=color,
                label=f"gamma={g:g} (~{half_width_km(g):.0f} km)")
    best = summary["cv_rmse"].min()
    ax.set_ylim(best * 0.98, best * 1.3)  # cut off the pairs that blow up, so the differences near the best stay visible
    ax.set_xscale("log")
    ax.set_xticks(sorted(summary["n_clusters"].unique()), labels=sorted(summary["n_clusters"].unique()))
    ax.minorticks_off()
    ax.set_xlabel("n_clusters (log scale)")
    ax.set_ylabel("10-fold CV RMSE ($)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(title="gamma (similarity half-width)", fontsize=8)
    fig.savefig(os.path.join(KM_TEST_DIR, filename), dpi=150, bbox_inches="tight")
    plt.show()

LIN_CLUSTERS = [20, 75, 150, 300, 600, 1200, 2000, 3000]
LIN_GAMMAS = [1, 5, 15, 50, 150, 500]

def score_linear(k, g):
    model = build_poly(keep_steps, ridge, degree=1, kmeans=True, gamma=g, n_clusters=k)
    return cross_validate(model, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)

lin_km = run_km_grid("linear_folds.csv", itertools.product(LIN_CLUSTERS, LIN_GAMMAS), score_linear)
lin_km_summary = km_summary(lin_km)
lin_km_summary.to_csv(os.path.join(KM_TEST_DIR, "linear_summary.csv"))
display(lin_km_summary)
display(km_heatmap(lin_km_summary))
km_plot(lin_km_summary, "Ridge (engineered + k-means): CV RMSE by n_clusters and gamma", "linear_plot.png")

In [ ]:
# ---------- TEST 2: n_clusters x gamma, neural network ----------
# Same preprocessing (prep_fit -> build, engineered + k-means), folds, early-stopping split, seeds and configuration
# (best_cfg) as cross_validate_nn; only n_clusters and gamma change. Note: best_cfg was tuned with n_clusters=75, gamma=15.
# Expensive: 10 folds x 3 seeds = 30 networks per pair. The grid can be narrowed around the best pairs of TEST 1.
NN_CLUSTERS = [75, 300, 1200, 3000]
NN_GAMMAS = [5, 15, 50, 150]
NN_SEEDS = (89, 233, 1597)  # same seeds as cross_validate_nn

def score_nn(k, g):
    steps, _ = feats[FINAL_FEATURE_SET]
    y = y_tr_raw
    scores = {m: [] for m in METRICS}
    for train_idx, val_idx in tqdm(list(cv.split(x_train)), desc=f"n_clusters={k}, gamma={g}", leave=False):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val = prep_fit(steps, x_train.iloc[fit_idx], y[fit_idx], x_train.iloc[stop_idx],
                                        x_train.iloc[val_idx], kmeans=True, gamma=g, n_clusters=k)
        fit_preds, val_preds = [], []
        for seed in NN_SEEDS:
            result = train_network(X_fit, y[fit_idx], X_stop, y[stop_idx], seed=seed, **best_cfg)
            fit_preds.append(predict(result, X_fit))
            val_preds.append(predict(result, X_val))
        fit_pred, val_pred = np.mean(fit_preds, axis=0), np.mean(val_preds, axis=0)
        scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        scores["test_r2"].append(r2_score(y[val_idx], val_pred))
    return {m: np.array(v) for m, v in scores.items()}

nn_km = run_km_grid("nn_folds.csv", itertools.product(NN_CLUSTERS, NN_GAMMAS), score_nn)
nn_km_summary = km_summary(nn_km)
nn_km_summary.to_csv(os.path.join(KM_TEST_DIR, "nn_summary.csv"))
display(nn_km_summary)
display(km_heatmap(nn_km_summary))
km_plot(nn_km_summary, "NN (engineered + k-means): CV RMSE by n_clusters and gamma", "nn_plot.png")